# Channel

> What happens to one transmission.

In [ ]:
#| default_exp channel

In [ ]:
#| export
# channel.py

from __future__ import annotations

import random
from dataclasses import replace
from abc import ABC, abstractmethod

from comm_core.core import Transmission, TransmissionResult
from comm_core.topology import Link


    

In [ ]:
#| export

class Channel(ABC):

    @abstractmethod
    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:
        pass

    def reset(self, seed: int | None = None) -> None:
        "Start over (e.g. a new episode); `seed` reseeds the channel's random generator, if it has one."
        pass

    def update(self, time: float) -> None:
        pass

In [ ]:
#| export

class IdentityChannel(Channel):
    """
    Perfect communication.
    """

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        message = transmission.message

        return TransmissionResult(
            success=True,
            transmission=transmission,
            payload=message.payload,
            latency=0.0,
            size_bits=message.size_bits or 0,
            snr_db=float("inf"),
            error_rate=0.0,
        )




In [ ]:
#| export

class AWGNChannel(Channel):
    """
    Simple analytical channel.

    This intentionally models only packet success/failure
    rather than individual bits.
    """

    def __init__(
        self,
        snr_db: float, # Signal-to-noise ratio (reported in the results)
        packet_error_rate: float = 0.0, # Probability that a packet is lost
        seed: int | None = None, # Seed of the channel's own random generator
    ):
        self.snr_db = snr_db
        self.packet_error_rate = packet_error_rate
        self.rng = random.Random(seed)

    def reset(self, seed: int | None = None) -> None:
        "`seed` reseeds the packet losses."
        if seed is not None:
            self.rng.seed(seed)

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        success = (
            self.rng.random()
            >= self.packet_error_rate
        )

        message = transmission.message

        return TransmissionResult(
            success=success,
            transmission=transmission,
            payload=(
                message.payload
                if success
                else None
            ),
            latency=0.0,
            size_bits=message.size_bits or 0,
            snr_db=self.snr_db,
            error_rate=self.packet_error_rate,
        )

## Fragmentation and retransmissions

`FragmentingChannel` delivers long messages the way a MAC layer does, over another channel (e.g. Wi-Fi's MTU and
retry limit, as in 802.11):

* **Packets:** a message is split into packets of at most `mtu_bits`.
* **Retries:** each packet is sent until it gets through, at most `retry_limit` times more.
* **Delivery:** the message arrives if every packet does, so a lost packet loses the message.
* **Time:** every attempt takes its airtime plus `overhead_s` (preamble, backoff, acknowledgement). The airtime is
  `rate_bps`'s, or the inner channel's latency. Each attempt is a transmission at its own time, so time-varying
  channels (e.g. `GilbertElliottFading`) see the state at that attempt.

The result's latency covers all attempts, and its metadata adds `packets`, `attempts` and `bits_on_air`.

The packets carry the message's size, not its bits. Use it over channels that decide from the size (e.g.
`SystemLevelChannel`, `AWGNChannel`), with `AnalyticalBackend`.

In [ ]:
#| export
class FragmentingChannel(Channel):
    "A MAC layer over another channel: packets of at most `mtu_bits`, each retried; the message arrives if every packet does."

    def __init__(
        self,
        channel: Channel, # The channel of each packet (deciding from its size, e.g. `SystemLevelChannel`)
        mtu_bits: int = 12_000, # Largest packet (1500 bytes)
        retry_limit: int = 7, # Retransmissions of a packet before the message is lost
        overhead_s: float = 0.0, # Time added to every attempt (preamble, backoff, acknowledgement)
        rate_bps: float | None = None, # Bit rate of the airtime; None: the inner channel's latency
    ):
        self.channel, self.mtu_bits, self.retry_limit = channel, mtu_bits, retry_limit
        self.overhead_s, self.rate_bps = overhead_s, rate_bps

    def reset(self, seed: int | None = None) -> None:
        self.channel.reset(seed)

    def update(self, time: float) -> None:
        self.channel.update(time)

    def set_positions(self, positions) -> None:
        "Where the nodes are, for an inner channel that depends on it."
        if hasattr(self.channel, 'set_positions'):
            self.channel.set_positions(positions)

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        message = transmission.message
        size = message.size_bits or 0
        packets = [self.mtu_bits] * (size // self.mtu_bits) + ([size % self.mtu_bits] if size % self.mtu_bits or not size else [])
        time, attempts, bits_on_air, energy, ok, last = transmission.start_time, 0, 0, 0.0, True, None
        for bits in packets:
            for _ in range(self.retry_limit + 1):
                packet = replace(transmission, message=replace(message, size_bits=bits), start_time=time, end_time=None)
                last = self.channel.transmit(packet, link)
                attempts, bits_on_air, energy = attempts + 1, bits_on_air + bits, energy + last.energy
                time += (bits / self.rate_bps if self.rate_bps else last.latency) + self.overhead_s
                if last.success:
                    break
            else:
                ok = False
                break
        return TransmissionResult(
            success=ok,
            transmission=transmission,
            payload=message.payload if ok else None,
            latency=time - transmission.start_time,
            size_bits=size,
            energy=energy,
            snr_db=last.snr_db,
            error_rate=last.error_rate,
            metadata={**last.metadata, 'packets': len(packets), 'attempts': attempts, 'bits_on_air': bits_on_air},
        )

### Tests

In [ ]:
from comm_core.core import Message, Transmission
from comm_core.topology import Link

In [ ]:
def losses(channel, n=200):
    t = Transmission(message=Message('a', 'b', payload=1, size_bits=8), sender='a', receiver='b', start_time=0.0)
    return [channel.transmit(t, Link('a', 'b')).success for _ in range(n)]

assert all(losses(IdentityChannel()))
first = losses(AWGNChannel(snr_db=5, packet_error_rate=0.3, seed=0))
assert first == losses(AWGNChannel(snr_db=5, packet_error_rate=0.3, seed=0))      # seeded
assert 0.2 < 1 - sum(first) / len(first) < 0.4
channel = AWGNChannel(snr_db=5, packet_error_rate=0.3, seed=1)
a = losses(channel)
channel.reset(seed=1)
assert losses(channel) == a                                                         # reset(seed) replays

In [ ]:
import numpy as np

In [ ]:
# 3 packets (1500 + 1500 + 500 bytes); a channel losing every packet 60% of the time
mac = FragmentingChannel(AWGNChannel(snr_db=5, packet_error_rate=0.6, seed=0), mtu_bits=12_000, retry_limit=7,
                         overhead_s=200e-6, rate_bps=65e6)
t = Transmission(message=Message('a', 'b', payload='image', size_bits=28_000), sender='a', receiver='b', start_time=1.0)
results = [mac.transmit(t, Link('a', 'b')) for _ in range(300)]
r = results[0]
assert r.metadata['packets'] == 3 and r.size_bits == 28_000 and r.metadata['attempts'] >= 3
assert r.metadata['bits_on_air'] >= 28_000
# airtime: every attempt's bits at 65 Mb/s, plus 200 µs each
assert all(abs(x.latency - (x.metadata['bits_on_air'] / 65e6 + 200e-6 * x.metadata['attempts'])) < 1e-12 for x in results)
# a packet gets through within 8 attempts with probability 1 - 0.6^8: the message with that, cubed
expected = (1 - 0.6 ** 8) ** 3
assert abs(np.mean([x.success for x in results]) - expected) < 0.05 and all(x.payload == 'image' for x in results if x.success)
# no retries: the message needs all 3 packets the first time
once = FragmentingChannel(AWGNChannel(5, 0.6, seed=1), retry_limit=0)
none = [once.transmit(t, Link('a', 'b')).success for _ in range(500)]
assert abs(np.mean(none) - 0.4 ** 3) < 0.04

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()